In [ ]:
import datetime
import glob
import matplotlib.pyplot as plt
from multiprocessing import Pool, Queue
import numpy as np
import os
import pickle
from subprocess import run, DEVNULL

from astropy.io import fits
from astropy.table import QTable
from astropy.wcs import WCS
import astropy.units as u

import sep

# Silence an annoying warning that pops up when using astrometry.net WCS FITS files
import logging
class SilenceFITSWarningFilter(logging.Filter):
    def filter(self, record):
        if 'The WCS transformation has more axes (2) than the image it is associated with (0)' in record.getMessage():
            return False
        return True
from astropy import log
log.addFilter(SilenceFITSWarningFilter())

%matplotlib widget

In [ ]:
# read_path = '/media/evanmayer/TIMCAM-FTS25-IMG/'
read_path = '/media/shared/'
# read_path = '/home/evanmayer/TIM_data/test_events/TIMcam/flight_test/'

write_path = '/home/evanmayer/TIM_data/test_events/TIMcam/flight_test/solve/'

In [ ]:
s = np.s_[::1]
files = sorted(glob.glob(read_path + 'img/saved_image_2025-10-01_1[8-9]*.fz', recursive=True))
files += sorted(glob.glob(read_path + 'img/saved_image_2025-10-01_2[0-3]*.fz', recursive=True))
files += sorted(glob.glob(read_path + 'img/saved_image_2025-10-02_0[0-4]*.fz', recursive=True))
files = files[s]
if not isinstance(files, list):
    files = [files,]

print(len(files))
print(files)

raw_columns = ['FOCUSMIN', 'FOCUS', 'FOCUSMAX', 'GAINFACT', 'EXPTIME', 'UTC-SEC', 'UTC-USEC', 'CCDTEMP']
qty = {col: [] for col in raw_columns}

raw_units = [u.dimensionless_unscaled, u.dimensionless_unscaled, u.dimensionless_unscaled, u.dimensionless_unscaled, u.s, u.s, 1e-6 * u.s, u.deg_C]
unit_lookup = {col: unit for col, unit in zip(raw_columns, raw_units)}

Build the raw data table - only stuff found in headers

In [ ]:
i = 0
n = len(files)
for file in files:
    i += 1
    print(f'progress: {100*i/n:.1f}%', end='\r')
    try:
        hdulist = fits.open(file, lazy_load_hdus=True)
        phdu, comphdu = hdulist
    except Exception as e:
        print(file, e)
        for key in raw_columns:
            qty[key].append(np.nan * unit_lookup[key])
        continue

    for key in raw_columns:
        qty[key].append(comphdu.header[key] * unit_lookup[key])

qty['FILES'] = files

img_table = QTable(qty)
try:
    img_table.write(read_path + 'tabulated.fits', overwrite=True)
except PermissionError as e:
    print(e, ', writing to local path', write_path)
img_table.write(write_path + 'tabulated.fits', overwrite=True)

# if overwriting results:
img_table_solved = img_table

# if updating on disk results:
# img_table_solved = QTable.read(write_path + 'tabulated_solved.fits')

Prepare to plate solve

In [ ]:
fetch_size = lambda fname: os.lstat(fname).st_size
fetch_timestamp = lambda utc_sec, utc_usec: utc_sec + utc_usec

In [ ]:
def solve_field(file, verbose=False):
    # Decompress file
    decomp_file = os.path.splitext(os.path.splitext(file)[0])[0] + '_funpack.fits'
    decomp_file = write_path + os.path.basename(decomp_file)
    if not os.path.exists(decomp_file):
        executable = 'funpack' # assumes astropy FITS tools in path
        options = [
            '-O', decomp_file, # specify output filename
        ]
        cmd = [executable,] + options + [file,]
        # print(cmd)
        # print(decomp_file)
        ret = run(cmd, stdout=DEVNULL)
        # ret.check_returncode()

    # Solve decompressed file
    executable = '/usr/local/astrometry/bin/solve-field'
    options = [
        '--dir=' + write_path,
        '--no-plots',
        # '--continue', # converse is '--overwrite'
        '--overwrite',
        # '--skip-solved',
        '--new-fits=none',
        '--parity=neg',
        '--code-tolerance=0.01',
        '--scale-low=6.7',
        '--scale-high=6.9',
        '--scale-units=arcsecperpix',
        '--use-source-extractor',
        '--objs=200',
        '--cpulimit=30',
    ]
    cmd = [executable, decomp_file] + options
    # print(cmd)
    if not verbose:
        ret = run(cmd, stderr=DEVNULL, stdout=DEVNULL)
    else:
        with open(write_path + 'solve_field_output.txt', 'w') as f:
            ret = run(cmd, stdout=f, stderr=f)
            print(ret.stdout, ret.stderr)
    # ret.check_returncode()

    # Look through astrometry.net output files
    solve_dir = write_path
    solved_file = solve_dir + os.path.splitext(os.path.basename(decomp_file))[0] + '.solved'

    if os.path.exists(solved_file):
        # get ra, dec, position angle
        wcs_file = os.path.splitext(solved_file)[0] + '.wcs'
        with fits.open(wcs_file) as f:
            h = f[0].header
        w = WCS(header=h)
        center = w.all_pix2world(h['IMAGEW'] / 2 + 0.5, h['IMAGEH'] / 2 + 0.5, 1)
        ra, dec = center[0], center[1]
        # https://github.com/dstndstn/astrometry.net/blob/039bafa8f1936735eb45213d7570ba03c3fa65e3/util/sip.c#L550
        det = w.wcs.cd[0, 0] * w.wcs.cd[1, 1] - w.wcs.cd[0, 1] * w.wcs.cd[1, 0]
        parity = (1 if det >= 0 else -1)
        T = parity * w.wcs.cd[0, 0] + w.wcs.cd[1, 1]
        A = parity * w.wcs.cd[1, 0] - w.wcs.cd[0, 1]
        orientation = -(np.atan2(A, T) * u.rad).to(u.deg).value

        # get plate scale
        # header = fits.open(wcs_file)[0].header
        for key, val in h.items():
            if 'COMMENT' not in key:
                continue
            if val.endswith('arcsec/pix'):
                scale = float(val.split(' ')[1])

        # calculate RMSE
        corr_file = os.path.splitext(solved_file)[0] + '.corr'
        tbl = fits.open(corr_file)[1].data
        rmse = scale * np.sqrt(np.mean((tbl.index_x - tbl.field_x)**2 + (tbl.index_y - tbl.field_y)**2))
        n_stars_matched = len(tbl.field_x)
        solved = 1
    else:
        ra = np.nan
        dec = np.nan
        orientation = np.nan
        scale = np.nan
        n_stars_found = np.nan
        n_stars_matched = np.nan
        rmse = np.nan
        solved = 0

    # axy file contains blobs sent to solver
    axy_file = os.path.splitext(solved_file)[0] + '.axy'
    tbl = fits.open(axy_file)[1].data
    try:
        n_stars_found = len(tbl.X) # default
    except AttributeError as e:
        n_stars_found = len(tbl.X_IMAGE) # source extractor

    # clean up
    os.remove(decomp_file)

    return ra, dec, orientation, scale, n_stars_found, n_stars_matched, rmse, solved

In [ ]:
# solve_field('/media/shared/img/saved_image_2025-10-02_02-00-58.fits.fz', verbose=True)
# hdulist = fits.open('/media/shared/img/saved_image_2025-10-02_02-00-58.fits.fz', lazy_load_hdus=True)
# _, comphdu = hdulist
# bkg = sep.Background(np.astype(comphdu.data, float))
# bkg_image = bkg.back()
# srcs = comphdu.data - bkg_image

# plt.close('all')
# fig, ax = plt.subplots(figsize=(12,5), ncols=2)
# # ax[0].imshow(bkg_image)
# m, s = srcs.mean(), srcs.std()
# ax[1].imshow(srcs, vmin=m-s, vmax=m+s)
# res = sep.extract(srcs, 3, err=4)
# print(len(res))
# ax[1].scatter(res['x'], res['y'], facecolor='none', edgecolor='w')

In [ ]:
def background_adu(file, plot=False):
    hdulist = fits.open(file, lazy_load_hdus=True)
    _, comphdu = hdulist
    bkg = sep.Background(np.astype(comphdu.data, float))
    bkg_image = bkg.back()

    if plot:
        fig, ax = plt.subplots()
        ax.imshow(bkg_image)

    return bkg.globalback

In [ ]:
derived_columns = ['SOLVE_ATTEMPTED', 'SOLVED', 'FILESIZE', 'TIMESTAMP', 'RA', 'DEC', 'ORIENTATION', 'SCALE', 'N_STARS_FOUND', 'N_STARS_MATCHED', 'RMSE', 'BKG_MED', 'BKG_MEAN']
derived_units = [u.dimensionless_unscaled, u.dimensionless_unscaled, u.dimensionless_unscaled, u.s, u.deg, u.deg, u.deg, u.arcsec / u.pix, u.dimensionless_unscaled, u.dimensionless_unscaled, u.arcsec, u.dimensionless_unscaled, u.dimensionless_unscaled]
unit_lookup.update({col: unit for col, unit in zip(derived_columns, derived_units)})

for col in derived_columns:
    img_table_solved[col] = np.ones_like(img_table_solved['FOCUS']).value * np.nan * unit_lookup[col]

n_prog = len(img_table_solved)
solution_progress = Queue()

In [ ]:
def solve_table_row(i):
    if img_table_solved['SOLVE_ATTEMPTED'][i] == 1:
        print('Skipping row, already attempted')
        return
    file = img_table_solved['FILES'][i]
    size = fetch_size(file)
    timestamp = fetch_timestamp(img_table_solved['UTC-SEC'][i], img_table_solved['UTC-USEC'][i])

    ra, dec, orientation, scale, n_stars_found, n_stars_matched, rmse, solved = solve_field(file)

    # print(ra, dec, orientation, scale, n_stars_found, n_stars_matched, rmse, solved)
    solution_packet = {
        'SOLVE_ATTEMPTED': 1,
        'FILES': file,
        'FILESIZE': size,
        'TIMESTAMP': timestamp.value,
        'RA': ra,
        'DEC': dec,
        'ORIENTATION': orientation,
        'SCALE': scale,
        'N_STARS_FOUND': n_stars_found,
        'N_STARS_MATCHED': n_stars_matched,
        'RMSE': rmse,
        'SOLVED': solved
    }
    pickle_back = write_path + os.path.splitext(os.path.splitext(os.path.basename(file))[0])[0] + '_sol.pickle'
    with open(pickle_back, 'wb') as f:
        pickle.dump(solution_packet, f)

    solution_progress.put(1)

    print(f'{file} {"SOLVED" if solved else "NOT SOLVED"} [{100 * solution_progress.qsize() / n_prog:.6f}%, {solution_progress.qsize()} / {n_prog}]', end='\r')
    return solution_packet


def sep_table_row(i):
    file = img_table_solved['FILES'][i]
    bkg_adu = background_adu(file)
    solution_packet = {
        'FILES': file,
        'BKG_MEAN': bkg_adu
    }
    pickle_back = write_path + os.path.splitext(os.path.splitext(os.path.basename(file))[0])[0] + '_back.pickle'
    with open(pickle_back, 'wb') as f:
        pickle.dump(solution_packet, f)

    solution_progress.put(1)
    
    print(f'{file} done. [{100 * solution_progress.qsize() / n_prog:.6f}%, {solution_progress.qsize()} / {n_prog}]', end='\r')
    return solution_packet

In [ ]:
# parallelize plate solving
solution_progress = Queue()
with Pool(processes=7) as pool:
    a_res = pool.map_async(solve_table_row, range(len(img_table_solved)))
    res = a_res.get()

In [ ]:
# a method involving pickled results, in case the above Pool never finishes
res = []
for file in glob.glob(write_path + '*_sol.pickle'):
    with open(file, 'rb') as f:
        res.append(pickle.load(f))

# unwind plate solving results
for j, result in enumerate(res):
    print(f'{j / len(res):.3f}', end='\r')
    match = (img_table_solved['FILES'] == result['FILES'])
    i = np.where(match)[0][0]
    for key, val in result.items():
        # print(i, key, val)
        if 'FILES' == key:
            img_table_solved[i][key] = val
        else:
            img_table_solved[i][key] = val * unit_lookup[key]

Use `sep` to estimate background mean values

In [ ]:
# parallelize background estimation
solution_progress = Queue()
with Pool(processes=7) as pool:
    a_res = pool.map_async(sep_table_row, range(len(img_table_solved)))
    res = a_res.get()

In [ ]:
# a method involving pickled results, in case the above Pool never finishes
res = []
for file in glob.glob(write_path + '*_back.pickle'):
    with open(file, 'rb') as f:
        res.append(pickle.load(f))

# unwind sep results
for j, result in enumerate(res):
    print(f'{j / len(res):.3f}', end='\r')
    match = (img_table_solved['FILES'] == result['FILES'])
    i = np.where(match)[0][0]
    for key, val in result.items():
        if 'FILES' == key:
            img_table_solved[i][key] = val
        else:
            img_table_solved[i][key] = val * unit_lookup[key]

In [ ]:
# img_table_solved.write(write_path + 'tabulated_solved.fits', overwrite=True)